# CDC PLACES EDA

This notebook:
1. **Loads** CDC PLACES data at the **ZCTA** (ZIP Code Tabulation Area) level using the **CDCPLACES R package**.
2. **Converts** the data to a **one-ZCTA-per-row** format (analysis-ready) in Python.
3. Runs **basic exploratory data analysis** (EDA) in Python.

Requires: R with `CDCPLACES` installed, and Python with `rpy2`, `pandas`, `matplotlib`, `seaborn`.

In [ ]:
# Python setup: rpy2 (R from Python), pandas, and EDA
import os
import tempfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rpy2.robjects import r, pandas2ri, default_converter
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr
from rpy2.robjects.vectors import StrVector

try:
    import geopandas as gpd
except ImportError:
    gpd = None

# Use this when converting R data.frames to pandas (see next cell):
#   with localconverter(default_converter + pandas2ri.converter):
#       df = pandas2ri.rpy2py(r_object)

print("Python and rpy2 ready.")

## 1. Load ZCTA data via R (CDCPLACES)

We use the CDCPLACES R package from Python via `rpy2`. Adjust `states` and `measures` to control which ZCTAs and health measures are pulled.

In [ ]:
# Load the CDCPLACES R package and fetch ZCTA-level data
# Ensure CDCPLACES is loaded in R (install in R first if needed: install.packages("CDCPLACES"))
cdcplaces = importr("CDCPLACES")

# Configuration: state(s) and measure(s) to pull
# For ZCTA, use state to filter (no county filter). Use measure = NULL to get all measures (slower).
states = ["CA"]  # e.g. ["CA"], ["CA", "NY"], or one state for faster runs
measures = ["CASTHMA", "COPD", "OBESITY", "DIABETES"]  # example; use None for all
release = "2023"

# Build R call: get_places(geography = "zcta", state = ..., measure = ..., release = ..., geometry = FALSE)
r.assign("states_r", StrVector(states))
r.assign("measures_r", StrVector(measures))
r.assign("release_r", release)

# get_places(geography = "zcta", state = c("CA"), measure = c("CASTHMA", "COPD", ...), release = "2023", geometry = FALSE)
r('places_zcta <- get_places(geography = "zcta", state = states_r, measure = measures_r, release = release_r, geometry = FALSE)')

# Avoid rpy2 conversion issues (coordinates/numpy): have R write CSV, Python reads it
tmp_csv = os.path.join(tempfile.gettempdir(), "places_zcta_r.csv")
r.assign("tmp_path", tmp_csv)
r('''
if ("sf" %in% class(places_zcta)) {
  if (requireNamespace("sf", quietly = TRUE)) places_zcta <- sf::st_drop_geometry(places_zcta)
}
# Keep only atomic columns (no list/geometry) so write.csv works
atomic <- sapply(places_zcta, is.atomic)
places_zcta <- places_zcta[, atomic, drop = FALSE]
write.csv(places_zcta, tmp_path, row.names = FALSE)
''')

places_long = pd.read_csv(tmp_csv)
try:
    os.remove(tmp_csv)
except OSError:
    pass

print("Raw data (long: one row per ZCTA × measure):")
print(places_long.shape)
places_long.head(10)

## 2. Reshape to one ZCTA per row

The API returns one row per (ZCTA, measure). We pivot so each measure becomes a column and each row is a single ZCTA.

In [ ]:
# Inspect column names (CDCPLACES may use slightly different names by release)
print("Columns:", list(places_long.columns))

# Detect column roles: ZCTA/geo id, measure id, and data value
def first_in(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

id_col = first_in(places_long.columns, ["LocationName", "ZCTA", "zcta", "locationname", "ZCTA5"])
measure_col = first_in(places_long.columns, ["MeasureId", "measureid", "Measure"])
value_col = first_in(places_long.columns, ["Data_Value", "data_value", "DataValue"])

# Fallback: if names differ, infer from position (long format often: geo, measure, value)
if id_col is None or measure_col is None or value_col is None:
    cols = list(places_long.columns)
    if id_col is None and len(cols) >= 1:
        id_col = cols[0]
    if measure_col is None and len(cols) >= 2:
        measure_col = cols[1]
    if value_col is None and len(cols) >= 3:
        value_col = cols[2]

print(f"Using: id_col={id_col!r}, measure_col={measure_col!r}, value_col={value_col!r}")

In [ ]:
# Pivot to one row per ZCTA; each measure becomes a column
# Keep state/geo identifiers if present for merging later
index_cols = [id_col]
if "StateAbbr" in places_long.columns:
    index_cols.insert(0, "StateAbbr")
elif "state_abbr" in places_long.columns:
    index_cols.insert(0, "state_abbr")

# Ensure we only use columns that exist
index_cols = [c for c in index_cols if c in places_long.columns]

places_wide = places_long.pivot_table(
    index=index_cols,
    columns=measure_col,
    values=value_col,
    aggfunc="first"  # one value per (ZCTA, measure)
).reset_index()

# Rename index-like column to a clear name for analysis
places_wide = places_wide.rename(columns={id_col: "ZCTA"}) if id_col != "ZCTA" else places_wide

print("One ZCTA per row (analysis-ready):")
print(places_wide.shape)
places_wide.head(10)

## 3. Basic EDA

Summary statistics, missing values, and distributions.

In [ ]:
# Summary: shape, dtypes, missing
print("Shape:", places_wide.shape)
print("\nDtypes:")
print(places_wide.dtypes)
print("\nMissing values per column:")
print(places_wide.isna().sum())
print("\nBasic describe (numeric):")
places_wide.describe()

In [ ]:
# Distributions: histograms for each measure (numeric columns only)
measure_cols = [c for c in places_wide.columns if c not in ["ZCTA", "StateAbbr", "state_abbr"] and places_wide[c].dtype in ["float64", "int64"]]
if not measure_cols:
    measure_cols = [c for c in places_wide.columns if places_wide[c].dtype in ["float64", "int64"]]

n = len(measure_cols)
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(4 * ((n + 1) // 2), 4 * 2))
if n == 1:
    axes = [axes]
else:
    axes = axes.flatten()
for i, col in enumerate(measure_cols):
    axes[i].hist(places_wide[col].dropna(), bins=30, edgecolor="black", alpha=0.7)
    axes[i].set_title(col)
    axes[i].set_xlabel("Value")
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.suptitle("Distribution of health measures (per ZCTA)", y=1.02)
plt.show()

In [ ]:
# Correlation matrix of health measures (if multiple numeric columns)
if len(measure_cols) >= 2:
    corr = places_wide[measure_cols].corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True)
    plt.title("Correlation between health measures (ZCTA-level)")
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 numeric measure columns for correlation plot.")

## 4. EDA: Asthma and COPD outcomes

Focusing on **asthma** (CASTHMA = current asthma among adults) and **COPD** (chronic obstructive pulmonary disease)—two respiratory outcomes often used as proxies for environmental health in pesticide-exposure research.

In [ ]:
# Subset to asthma and COPD columns (ensure they exist)
resp_cols = [c for c in ["CASTHMA", "COPD"] if c in places_wide.columns]
resp_df = places_wide[["ZCTA"] + resp_cols].dropna() if resp_cols else pd.DataFrame()
if len(resp_cols) < 2:
    print("CASTHMA or COPD not in places_wide. Columns:", list(places_wide.columns))
else:
    print("Asthma (CASTHMA) and COPD — summary by ZCTA:")
    print(resp_df[resp_cols].describe())
    print("\nCorrelation (asthma vs COPD):", resp_df["CASTHMA"].corr(resp_df["COPD"]).round(3))

In [ ]:
# Scatter: Asthma vs COPD (ZCTA-level)
if len(resp_cols) == 2:
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(resp_df["CASTHMA"], resp_df["COPD"], alpha=0.4, s=15)
    z = np.polyfit(resp_df["CASTHMA"], resp_df["COPD"], 1)
    p = np.poly1d(z)
    x_line = np.linspace(resp_df["CASTHMA"].min(), resp_df["CASTHMA"].max(), 100)
    ax.plot(x_line, p(x_line), "r-", linewidth=2, label="linear fit")
    ax.set_xlabel("Current asthma among adults (%)")
    ax.set_ylabel("COPD (%)")
    ax.set_title("Asthma vs COPD by ZCTA (California)")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution comparison: side-by-side boxplots
if len(resp_cols) == 2:
    fig, ax = plt.subplots(figsize=(5, 4))
    resp_df[resp_cols].boxplot(ax=ax)
    ax.set_ylabel("Prevalence (%)")
    ax.set_title("Asthma and COPD prevalence by ZCTA")
    plt.tight_layout()
    plt.show()

In [ ]:
# ZCTAs in top quartile for both asthma and COPD (high respiratory burden)
if len(resp_cols) == 2:
    q75_a = resp_df["CASTHMA"].quantile(0.75)
    q75_c = resp_df["COPD"].quantile(0.75)
    high_both = resp_df[(resp_df["CASTHMA"] >= q75_a) & (resp_df["COPD"] >= q75_c)]
    print(f"ZCTAs in top quartile for BOTH asthma and COPD: {len(high_both)}")
    print(f"  Asthma ≥ {q75_a:.1f}%, COPD ≥ {q75_c:.1f}%")
    print("\nExample ZCTAs (high respiratory burden):")
    print(high_both.nlargest(10, "CASTHMA")[["ZCTA", "CASTHMA", "COPD"]])

In [ ]:
# Optional: save analysis-ready data for use in other scripts
# out_path = "places_zcta_one_per_row.csv"
# places_wide.to_csv(out_path, index=False)
# print(f"Saved to {out_path}")

# Final view: analysis-ready dataframe (one ZCTA per row)
places_wide

## 5. State-level data for COPD and Asthma

PLACES does not provide state as a geography. To get **state-level** estimates, we pull **county**-level data for CASTHMA and COPD, then aggregate by state (e.g. mean across counties). Run the cell below to fetch county data and build a one-row-per-state table.

In [ ]:
# Pull COUNTY-level PLACES data for COPD and Asthma (then we aggregate to state)
cdcplaces = importr("CDCPLACES")
r.assign("measures_r", StrVector(["CASTHMA", "COPD"]))
r.assign("release_r", "2023")

# geography = "county", state = NULL gets all US counties (or specify states_r for a subset)
r('places_county <- get_places(geography = "county", state = NULL, measure = measures_r, release = release_r, geometry = FALSE)')

# Keep only atomic columns and write to CSV (avoids rpy2 conversion issues)
tmp_csv = os.path.join(tempfile.gettempdir(), "places_county_r.csv")
r.assign("tmp_path", tmp_csv)
r('''
atomic <- sapply(places_county, is.atomic)
places_county <- places_county[, atomic, drop = FALSE]
write.csv(places_county, tmp_path, row.names = FALSE)
''')

county_df = pd.read_csv(tmp_csv)
try:
    os.remove(tmp_csv)
except OSError:
    pass

# Standardize column names (API may return mixed case)
county_df.columns = [c.strip().lower().replace(" ", "_") for c in county_df.columns]
# Common names: stateabbr, locationname, measureid, data_value
state_col = "stateabbr" if "stateabbr" in county_df.columns else [c for c in county_df.columns if "state" in c][0]
loc_col = "locationname" if "locationname" in county_df.columns else [c for c in county_df.columns if "location" in c or "county" in c][0]
measure_col = "measureid" if "measureid" in county_df.columns else [c for c in county_df.columns if "measure" in c][0]
value_col = "data_value" if "data_value" in county_df.columns else [c for c in county_df.columns if "value" in c and "conf" not in c][0]

print("County-level data shape:", county_df.shape)
county_df.head()

In [ ]:
# Reshape to one row per county (CASTHMA and COPD as columns); keep locationid (FIPS) for mapping
id_col = "locationid" if "locationid" in county_df.columns else None
pivot_index = [state_col, loc_col] + ([id_col] if id_col else [])
county_wide = county_df.pivot_table(
    index=pivot_index,
    columns=measure_col,
    values=value_col,
    aggfunc="first"
).reset_index()

# State-level: mean prevalence across counties in each state
state_level = county_wide.groupby(state_col)[["CASTHMA", "COPD"]].mean().reset_index()
state_level.columns = ["State", "CASTHMA", "COPD"]

print("State-level COPD and Asthma (mean prevalence across counties):")
print(state_level.shape)
display(state_level.sort_values("CASTHMA", ascending=False).head(15))

## 6. County-level maps (COPD and Asthma)

Choropleth maps of county-level prevalence. We merge PLACES data with US Census county boundaries (FIPS) and plot with `geopandas`. The data are **single-year** (the API returns a `year` column, e.g. 2019 for the 2021 PLACES release)—not cross-year.

In [ ]:
# County maps: need geopandas and US county boundaries
# %pip install geopandas -q

# US counties GeoJSON: FIPS code identifies each county (5-digit: state + county)
if gpd is not None:
        COUNTIES_URL = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
        counties_gdf = gpd.read_file(COUNTIES_URL)
        # Normalize FIPS to 5-digit string (GeoJSON "id" can be string "01001" or number 12001/12001.0)
        def to_fips5(x):
            if pd.isna(x): return ""
            try: return str(int(float(x))).zfill(5)
            except (ValueError, TypeError): return str(x).replace(".0", "").zfill(5)
        id_series = counties_gdf["id"] if "id" in counties_gdf.columns else counties_gdf.index.astype(str)
        counties_gdf["GEOID"] = id_series.apply(to_fips5)

        # PLACES county_wide: ensure we have FIPS (5-digit locationid) for merge
        county_wide = county_wide.copy()
        if "locationid" not in county_wide.columns and "locationid" in county_df.columns:
            # Pivot was run before locationid was in the index: attach FIPS from county_df
            sk = "stateabbr" if "stateabbr" in county_wide.columns else "StateAbbr"
            lk = "locationname" if "locationname" in county_wide.columns else "LocationName"
            fips_lookup = county_df[[sk, lk, "locationid"]].drop_duplicates()
            county_wide = county_wide.merge(fips_lookup, on=[sk, lk], how="left")
            county_wide["FIPS"] = county_wide["locationid"].apply(to_fips5)

        # Merge prevalence onto geography
        merged = counties_gdf.merge(
        county_wide[["FIPS", "CASTHMA", "COPD"]].drop_duplicates("FIPS"),
        left_on="GEOID",
        right_on="FIPS",
        how="inner"
        )
        merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=counties_gdf.crs)

        # Continental US only (exclude AK, HI, territories) for clearer map
        merged_cont = merged[~merged["GEOID"].str.startswith(("02", "15", "72", "78", "60", "66", "69"))].copy()

        # Data year: PLACES API returns a year column (e.g. 2021 for 2023 release); single year per pull
        data_year = int(county_df["year"].iloc[0]) if "year" in county_df.columns else None
        print("Counties with PLACES data (merged):", len(merged_cont), "| Data year:", data_year if data_year else "N/A")
        merged_cont.head(2)

else:
    print("Install geopandas for county maps: pip install geopandas")

In [ ]:
# Choropleth maps: COPD and Asthma prevalence by county
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Shared scale for comparison (percent)
vmin = min(merged_cont["CASTHMA"].min(), merged_cont["COPD"].min())
vmax = max(merged_cont["CASTHMA"].max(), merged_cont["COPD"].max())

merged_cont.plot(ax=axes[0], column="CASTHMA", legend=True, cmap="YlOrRd", edgecolor="none",
                 legend_kwds={"label": "Current asthma among adults (%)", "shrink": 0.6})
axes[0].set_title("County-level asthma prevalence (CASTHMA)")
axes[0].axis("off")

merged_cont.plot(ax=axes[1], column="COPD", legend=True, cmap="YlOrRd", edgecolor="none",
                 legend_kwds={"label": "COPD prevalence (%)", "shrink": 0.6})
axes[1].set_title("County-level COPD prevalence")
axes[1].axis("off")

year_label = f" ({data_year})" if data_year else ""
plt.suptitle(f"CDC PLACES: County-level respiratory outcomes (continental US){year_label}", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---

**Next steps:** Use `places_wide` in other analyses (e.g. merge with pesticide exposure by ZCTA). Uncomment the save cell above to export `places_zcta_one_per_row.csv`. To pull more states or measures, edit `states` and `measures` in the first data-load cell and re-run.